# Noyau gaussien ou caractéristiques apprises

Ce notebook accompagne l'exercice 5.4. Les données ont la forme de deux demi-lunes entrelacées : un score affine est trop rigide, mais deux constructions non affines deviennent possibles.

Le SVM gaussien fixe un noyau puis résout un problème convexe. Le MLP apprend sa représentation en même temps que son classificateur final. Nous comparerons leurs frontières, leurs erreurs et leur sensibilité aux hyperparamètres.

## Parcours

1. [Deux demi-lunes et une partition déterministe](#demi-lunes)
2. [SVM affine et SVM gaussien](#svm-noyau)
3. [MLP et caractéristiques apprises](#mlp-lunes)
4. [Frontières et complexités](#comparaison-lunes)
5. [Perturbations déterministes](#perturbations-lunes)

In [1]:
import jax

jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import optax
import flax
from flax import nnx
import sklearn
from sklearn.svm import SVC

print(
    f"JAX {jax.__version__}, Flax {flax.__version__}, "
    f"Optax {optax.__version__}, scikit-learn {sklearn.__version__}"
)

JAX 0.11.1, Flax 0.12.9, Optax 0.2.8, scikit-learn 1.9.0


<a id="demi-lunes"></a>
## 1. Deux demi-lunes et une partition déterministe

Pour $\theta_i=i\pi/(m-1)$, construire
$$
x_i^+=(\cos\theta_i,\sin\theta_i),\qquad
x_i^-=(1-\cos\theta_i,1/2-\sin\theta_i).
$$

Utiliser les indices modulo $5$ pour obtenir une partition reproductible : trois indices pour l'apprentissage, un pour la validation et un pour le test. La même règle doit être appliquée aux deux classes.

In [ ]:
# À compléter.

In [ ]:
# À compléter.

<a id="svm-noyau"></a>
## 2. SVM affine et SVM gaussien

Nous utilisons la classe `SVC` de `scikit-learn`, dont l'implémentation repose sur la bibliothèque LIBSVM.

1. Ajuster un SVM affine avec `SVC(kernel="linear")`.
2. Pour le noyau gaussien
   $$K_\gamma(x,x')=\exp(-\gamma\lVert x-x'\rVert^2),$$
   choisir $C$ et $\gamma$ en minimisant la fréquence d'erreur de validation.
3. Ne jamais utiliser les données de test pour effectuer ce choix.
4. Représenter les vecteurs supports du modèle gaussien retenu.

In [ ]:
# À compléter.

<a id="mlp-lunes"></a>
## 3. MLP et caractéristiques apprises

La classe et la fonction d'apprentissage suivantes sont fournies. Ajuster plusieurs largeurs et plusieurs initialisations. Retenir la machine d'après les données de validation, comme pour le noyau.

La pénalisation porte ici sur l'ensemble des paramètres du MLP.

In [5]:
class MLPBinaire(nnx.Module):
    def __init__(self, dimension, largeur, *, rngs):
        self.affine_1 = nnx.Linear(dimension, largeur, rngs=rngs)
        self.affine_2 = nnx.Linear(largeur, largeur, rngs=rngs)
        self.affine_3 = nnx.Linear(largeur, 1, rngs=rngs)

    def __call__(self, x):
        y = jnp.tanh(self.affine_1(x))
        y = jnp.tanh(self.affine_2(y))
        return self.affine_3(y)[..., 0]

def norme_parametres(machine):
    return sum(
        jnp.sum(feuille**2)
        for feuille in jax.tree.leaves(nnx.state(machine, nnx.Param))
    )

def cout_mlp(machine, x, z, lamb):
    return (
        jnp.mean(jax.nn.softplus(-z * machine(x)))
        + 0.5 * lamb * norme_parametres(machine)
    )

@nnx.jit
def pas_mlp(machine, optimiseur, x, z, lamb):
    fonction = lambda m: cout_mlp(m, x, z, lamb)
    valeur, gradient = nnx.value_and_grad(fonction)(machine)
    optimiseur.update(machine, gradient)
    return valeur

def entrainer_mlp(x, z, largeur, graine, *, lamb=1.0e-4, iterations=1500):
    machine = MLPBinaire(x.shape[1], largeur, rngs=nnx.Rngs(graine))
    optimiseur = nnx.Optimizer(
        machine, optax.adam(1.0e-2), wrt=nnx.Param
    )
    x_jax = jnp.asarray(x)
    z_jax = jnp.asarray(z)
    for _ in range(iterations):
        valeur = pas_mlp(machine, optimiseur, x_jax, z_jax, lamb)
    return machine, float(valeur)

In [ ]:
# À compléter.

<a id="comparaison-lunes"></a>
## 4. Frontières et complexités

Représenter les trois frontières sur le même domaine. Comparer :

- les fréquences d'erreur sur apprentissage, validation et test ;
- le nombre de vecteurs supports du SVM gaussien ;
- le nombre de paramètres du MLP ;
- la sensibilité de chaque construction à ses hyperparamètres.

Le nombre de vecteurs supports et le nombre de paramètres ne sont pas deux mesures directement comparables de la complexité effective.

In [7]:
g1 = np.linspace(-1.4, 2.4, 320)
g2 = np.linspace(-1.0, 1.4, 260)
G1, G2 = np.meshgrid(g1, g2)
grille = np.column_stack([G1.ravel(), G2.ravel()])

def tracer_modele(axe, score, titre, supports=None):
    valeurs = np.asarray(score(grille)).reshape(G1.shape)
    axe.contourf(G1, G2, valeurs, levels=[-100, 0, 100], colors=["C0", "C3"], alpha=0.15)
    axe.contour(G1, G2, valeurs, levels=[0.0], colors="k")
    for etiquette, couleur, marqueur in [(-1, "C0", "o"), (1, "C3", "s")]:
        masque = z_app == etiquette
        axe.scatter(x_app[masque, 0], x_app[masque, 1], c=couleur, marker=marqueur, s=18)
    if supports is not None:
        axe.scatter(
            supports[:, 0], supports[:, 1],
            s=65, facecolors="none", edgecolors="k", label="vecteurs supports"
        )
        axe.legend(loc="lower right", fontsize=8)
    axe.set_title(titre)
    axe.set(xlabel=r"$x_1$", ylabel=r"$x_2$")
    axe.grid(True)

In [ ]:
# À compléter.

In [ ]:
# À compléter.

<a id="perturbations-lunes"></a>
## 5. Perturbations déterministes

Ajouter aux deux demi-lunes des perturbations déterministes d'amplitude croissante, par exemple des combinaisons de $\sin(7\theta)$ et $\cos(11\theta)$. Reprendre la comparaison pour quelques amplitudes.

La frontière la plus flexible donne-t-elle toujours la plus petite fréquence d'erreur sur les données de test ? Pourquoi la sélection sur les données d'apprentissage seules serait-elle trompeuse ?

In [ ]:
# À compléter.

## Bilan

Un SVM à noyau et un MLP construisent tous deux des frontières non affines, mais ils n'apprennent pas les mêmes objets. Le premier choisit un classificateur dans un espace de caractéristiques fixé ; le second apprend aussi cet espace. Comparer leurs résultats exige une même partition et une même règle de sélection.